# 04 - Quantitative Metrics and Performance Evaluation
This notebook runs inference over the entire test set to calculate standard classification metrics (Accuracy, F1-Score) and localization metrics (3-Way mIoU). It also includes a False Positive analysis to verify the model's behavior on clean images, and maps classification confidence against localization accuracy.

*Note: This notebook generates the `df_diag.pkl` state file required for the final diagnostic analysis.*

In [ ]:
# Cell 8: Quantitative Metrics[cite: 1]
img_dir = Path(CFG["test_dir"]) / 'images/before'
mask_dir = Path(CFG["test_dir"]) / 'masks/before'
img_paths = list(img_dir.rglob('*.jpg')) + list(img_dir.rglob('*.png'))

all_preds, all_labels = [], []
model.eval()

with torch.no_grad():
    for inputs, labels in tqdm(test_loader, desc="Testing Classifier"):
        inputs, labels = inputs.to(CFG["device"]), labels.to(CFG["device"])
        outputs = model(inputs)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

scores = {"Manual_CAM": [], "Grad_CAM": [], "Grad_CAM_Plus": []}

for path in tqdm(img_paths, desc="Evaluating 3-Way Localization"):
    img_pil = Image.open(path).convert('RGB')
    input_tensor = eval_transforms(img_pil).unsqueeze(0).to(CFG["device"])

    with torch.no_grad():
        output = model(input_tensor)
        confidence = F.softmax(output, dim=1)[0][BEFORE_IDX].item()
    
    h_man, h_grad, h_plus = get_all_heatmaps(input_tensor, BEFORE_IDX)

    m_man  = get_final_mask(h_man,  model_confidence=confidence)
    m_grad = get_final_mask(h_grad, model_confidence=confidence)
    m_plus = get_final_mask(h_plus, model_confidence=confidence)
    
    rel_path = path.relative_to(img_dir)
    gt_path = mask_dir / rel_path.with_suffix('.png')
    if not gt_path.exists(): 
        gt_path = mask_dir / rel_path.with_suffix('.jpg')
    
    if gt_path.exists():
        gt_mask = np.array(Image.open(gt_path).convert('L').resize((CFG["img_size"], CFG["img_size"]), Image.NEAREST)) > 0
        
        scores["Manual_CAM"].append(calculate_iou(m_man > 0, gt_mask))
        scores["Grad_CAM"].append(calculate_iou(m_grad > 0, gt_mask))
        scores["Grad_CAM_Plus"].append(calculate_iou(m_plus > 0, gt_mask))

acc = accuracy_score(all_labels, all_preds)
f1 = f1_score(all_labels, all_preds)

print("\n" + "="*60)
print("             FINAL SYSTEM REPORT")
print("="*60)
print(f"Classification Accuracy: {acc*100:.2f}%")
print(f"F1-Score (Waste):        {f1*100:.2f}%")
print("-" * 60)

iou_array = np.array(scores['Grad_CAM_Plus'])
num_samples = len(iou_array)

print("LOCALIZATION PERFORMANCE (3-WAY COMPARISON):")
print(f"*Note: These results are averaged over the full test set")
print(f"       ({num_samples} samples), providing a statistically robust")
print(f"       evaluation beyond the visual examples in Cell 7.*")
print("\nMean IoU Comparison:")
print(f"  Manual weight-based CAM: {np.mean(scores['Manual_CAM']):.4f}")
print(f"  Standard Grad-CAM:       {np.mean(scores['Grad_CAM']):.4f}")
print(f"  Grad-CAM++ (Ours):       {np.mean(iou_array):.4f}")

cm = confusion_matrix(all_labels, all_preds)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Clean', 'Waste'])
disp.plot(cmap='Blues')
plt.title('Test Set Confusion Matrix')
plt.show()

In [ ]:
# Cell 8.1: False Positive Analysis[cite: 1]
after_img_dir = Path(CFG["test_dir"]) / 'images/after'
after_img_paths = list(after_img_dir.rglob('*.jpg'))

fp_mask_coverages = []
for path in tqdm(after_img_paths[:200], desc="FP Analysis on Clean Images"):
    img_pil = Image.open(path).convert('RGB')
    input_tensor = eval_transforms(img_pil).unsqueeze(0).to(CFG["device"])
    with torch.no_grad():
        out = model(input_tensor)
        conf = F.softmax(out, dim=1)[0][BEFORE_IDX].item()
    h = grad_cam_plus_engine(input_tensor=input_tensor,
                              targets=[ClassifierOutputTarget(BEFORE_IDX)])[0, :]
    h = (h - h.min()) / (h.max() - h.min() + 1e-8)
    mask = get_final_mask(h, model_confidence=conf)
    fp_mask_coverages.append((mask > 0).mean())

print(f"Mean mask coverage on CLEAN images: {np.mean(fp_mask_coverages)*100:.2f}%")
print(f"% clean images with non-empty mask: {(np.array(fp_mask_coverages) > 0).mean()*100:.1f}%")

In [ ]:
# Cell 8.2 & 8.5: Diagnostic Analysis & State Export[cite: 1]
print("\n--- HONEST PERFORMANCE SUMMARY ---")
print(f"Median IoU (Grad-CAM++):         {np.median(scores['Grad_CAM_Plus']):.4f}")
print(f"Mean IoU (Grad-CAM++):           {np.mean(scores['Grad_CAM_Plus']):.4f}")
print(f"% samples with IoU > 0.3:        {(np.array(scores['Grad_CAM_Plus']) > 0.3).mean()*100:.1f}%")

diagnostic_data = []
model.eval()

for path in tqdm(img_paths, desc="Processing Diagnostics"):
    img_pil = Image.open(path).convert('RGB')
    input_tensor = eval_transforms(img_pil).unsqueeze(0).to(CFG["device"])
    
    with torch.no_grad():
        output = model(input_tensor)
        probabilities = F.softmax(output, dim=1)
        confidence = probabilities[0][BEFORE_IDX].item() 
    
    h_plus = grad_cam_plus_engine(input_tensor=input_tensor, targets=[ClassifierOutputTarget(BEFORE_IDX)])[0, :]
    h_plus = cv2.GaussianBlur(h_plus, CFG["smooth_kernel"], 0)
    h_plus = (h_plus - h_plus.min()) / (h_plus.max() - h_plus.min() + 1e-8)
    
    final_mask = get_final_mask(h_plus, model_confidence=confidence)
    
    rel_path = path.relative_to(img_dir)
    gt_path = mask_dir / rel_path.with_suffix('.png')
    if not gt_path.exists(): 
        gt_path = mask_dir / rel_path.with_suffix('.jpg')
    
    iou = 0.0
    if gt_path.exists():
        gt_mask_pil = Image.open(gt_path).convert('L').resize((CFG["img_size"], CFG["img_size"]), Image.NEAREST)
        gt_mask = (np.array(gt_mask_pil) > 0).astype(np.uint8)
        iou = calculate_iou(final_mask > 0, gt_mask)
    
    diagnostic_data.append({
        "Filepath": str(path),
        "Filename": path.name,
        "Confidence": confidence,
        "IoU": iou
    })

df_diag = pd.DataFrame(diagnostic_data)

plt.figure(figsize=(10, 6))
sns.regplot(data=df_diag, x="Confidence", y="IoU", scatter_kws={'alpha':0.5, 'color':'#4C72B0'}, line_kws={'color':'#C44E52'})
plt.title("Diagnostic: Does higher classification confidence lead to better IoU?", fontsize=14, fontweight='bold')
plt.xlabel("Model Confidence (Probability of 'Waste')")
plt.ylabel("Localization Accuracy (IoU)")
plt.grid(True, alpha=0.3)
plt.show()

# Export state explicitly for the final diagnostics notebook
df_diag.to_pickle('df_diag.pkl')
print("Successfully preserved diagnostic state to 'df_diag.pkl'.")